In [7]:
# notebooks/05_sar_drafter.ipynb
# Day 18 — SAR Drafter Testing + Output Quality Review
# Run after shap_narratives.py and sar_drafter.py are confirmed complete.
# Requires: shap_values, y_prob_test, y_test, df_test, feat_cols
# from your Day 15 SHAP analysis in memory (or reload below).

import numpy as np
import pandas as pd
import json
import joblib
import os
import sys
sys.path.insert(0, os.path.abspath(".."))  # or wherever your actual sar_drafter.py lives
from shap_narratives import build_shap_dict
from sar_drafter import draft_sar_narrative

# ---------------------------------------------------------------------------
# Reload artifacts if running fresh (skip if already in memory from Day 15)
# ---------------------------------------------------------------------------
model       = joblib.load('../../models/fraud_model.pkl')
feat_cols   = joblib.load('../../models/feature_cols.pkl')
threshold   = joblib.load('../../models/threshold.pkl')
explainer   = joblib.load('../../models/shap_explainer.pkl')
shap_values = joblib.load('../../models/shap_values_sample.pkl')
y_test=joblib.load("../../models/y_test.pkl")
y_prob_test=joblib.load("../../models/y_prob_test.pkl")
df_test=pd.read_csv("../../data/processed/test_engineered.csv")
# ---------------------------------------------------------------------------
# NOTE: shap_values_sample.pkl was saved on 2000 test rows.
# y_prob_test and y_test below must index into that same 2000-row slice.
# If you saved the full test set probs separately, adjust accordingly.
# ---------------------------------------------------------------------------
# Function to help wrap the investigation brief
import textwrap

def print_brief(text, width=80):
    for para in text.split('\n'):
        print(textwrap.fill(para, width=width) if para.strip() else '')
# ---------------------------------------------------------------------------
# Grouped SHAP display helper — purely presentational, no data changes
# ---------------------------------------------------------------------------
from collections import OrderedDict
import textwrap as _tw

def _print_shap_grouped(top_features: list, narrative_width: int = 76) -> None:
    """
    Display risk-increasing SHAP features grouped by risk category.
    Narrative and FATF reference are printed once per category, not per feature.
    Sort order (descending abs_impact) is preserved within each group.
    """
    positive = [f for f in top_features if f.get('shap_value', 0) > 0]

    if not positive:
        print("  No dominant risk-increasing factors identified.")
        return

    groups: OrderedDict = OrderedDict()
    for f in positive:
        cat = f.get('category', 'Unknown')
        groups.setdefault(cat, []).append(f)

    for category, features in groups.items():
        count = len(features)
        label = f"{count} contributing feature{'s' if count > 1 else ''}"
        print(f"\n  {category} ({label})\n")

        for f in features:
            print(f"      [{f['shap_value']:+.5f}]  {f['feature']}")

        rep = features[0]
        print("\n  Narrative:")
        for line in _tw.wrap(rep['narrative'], width=narrative_width):
            print(f"  {line}")
        print("\n  FATF:")
        for line in _tw.wrap(rep['fatf_ref'], width=narrative_width):
            print(f"  {line}")
        print()


# ---------------------------------------------------------------------------
# Define test indices across the 3 expected tier categories
# ---------------------------------------------------------------------------

# 5 confirmed fraud cases
fraud_indices = np.where(y_test.values[:2000] == 1)[0][:5]

# 3 borderline cases — ELEVATED RISK territory (score 0.35-0.65)
# Adjust bounds if your threshold means borderline lands elsewhere
borderline_mask    = (y_prob_test[:2000] > 0.35) & (y_prob_test[:2000] < 0.65)
borderline_indices = np.where(borderline_mask)[0][:3]

# 2 legitimate low-risk cases (score < 0.15, actual = 0)
legit_mask    = (y_prob_test[:2000] < 0.15) & (y_test.values[:2000] == 0)
legit_indices = np.where(legit_mask)[0][:2]

test_indices = list(dict.fromkeys(
    list(fraud_indices)
    + list(borderline_indices)
    + list(legit_indices)
))

print(f"Test set: {len(fraud_indices)} fraud | "
      f"{len(borderline_indices)} borderline | "
      f"{len(legit_indices)} legit")
print(f"Total: {len(test_indices)} cases\n")

# ---------------------------------------------------------------------------
# Run the full SHAP → narrative pipeline on each case
# ---------------------------------------------------------------------------

results = []

for idx in test_indices:
    shap_row = shap_values[idx]                        # 1D SHAP array
    prob     = float(y_prob_test[idx])
    txn_id   = str(df_test.iloc[idx]['TransactionID'])
    actual   = int(y_test.iloc[idx])

    shap_dict = build_shap_dict(txn_id, prob, shap_row, feat_cols)
    narrative = draft_sar_narrative(shap_dict)

    results.append({
        'txn_id':    txn_id,
        'prob':      round(prob, 4),
        'tier':      narrative['risk_tier'],
        'actual':    'FRAUD' if actual else 'LEGIT',
        'shap_dict': shap_dict,
        'narrative': narrative,
    })

# ---------------------------------------------------------------------------
# Print full output for manual checklist review
# ---------------------------------------------------------------------------

SEPARATOR = "=" * 72

for r in results:
    print(f"\n{SEPARATOR}")
    print(f"Transaction    : {r['txn_id']}")
    print(f"Score          : {r['prob']:.4f}   |   Actual: {r['actual']}")
    print(f"Risk Tier      : {r['tier']}")
    print(f"Narrative source: {r['narrative']['narrative_source']}")   # 'llm' or 'fallback'
    print(f"LLM model used : {r['narrative']['llm_model']}")

    print(f"\nRisk themes    : {r['shap_dict']['risk_themes']}")

    print(f"\nTop SHAP factors (risk-increasing only):")
    _print_shap_grouped(r['shap_dict']['top_features'])

    print(f"\nINVESTIGATION BRIEF ({r['narrative']['narrative_source'].upper()}):")
    print_brief(r['narrative']['investigation_brief'])

    if r['narrative']['compliance_notice']:
        print(f"\nCOMPLIANCE NOTICE (deterministic):")
        print(r['narrative']['compliance_notice'])
        # print(r['narrative']['compliance_notice'][:500] + "...")
    else:
        print(f"\nCOMPLIANCE NOTICE: None ({r['tier']} — not required)")

# ---------------------------------------------------------------------------
# Summary counts
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print(f"Total cases tested : {len(results)}")
print(f"HIGH RISK          : {sum(1 for r in results if r['tier'] == 'HIGH RISK')}")
print(f"ELEVATED RISK      : {sum(1 for r in results if r['tier'] == 'ELEVATED RISK')}")
print(f"LOW RISK           : {sum(1 for r in results if r['tier'] == 'LOW RISK')}")
print(f"LLM-generated      : {sum(1 for r in results if r['narrative']['narrative_source'] == 'llm')}")
print(f"Fallback briefs    : {sum(1 for r in results if r['narrative']['narrative_source'] == 'fallback')}")

# ---------------------------------------------------------------------------
# Automated assertion checks — mirrors the quality checklist
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print("AUTOMATED ASSERTION CHECKS\n")

assertion_failures = []

for r in results:
    txn = r['txn_id']
    tier = r['tier']
    notice = r['narrative']['compliance_notice']
    brief  = r['narrative']['investigation_brief']

    # 1. HIGH RISK must have POCA 2002 s.330 language in compliance_notice
    if tier == 'HIGH RISK':
        checks = {
            'POCA 2002 in notice':               'POCA 2002' in (notice or ''),
            'as soon as is practicable':         'as soon as is practicable' in (notice or ''),
            'UKFIU/NCA in notice':               'National Crime Agency' in (notice or ''),
            'does not confirm in notice':        'does not confirm' in (notice or ''),
            'notice is not None':                notice is not None,
        }
        for check, passed in checks.items():
            if not passed:
                assertion_failures.append(f"[{txn}] HIGH RISK failed: {check}")
                print(f"  ✗ [{txn}] {check}")
            else:
                print(f"  ✓ [{txn}] {check}")

    # 2. ELEVATED RISK must have analyst review language + POCA mention
    elif tier == 'ELEVATED RISK':
        checks = {
            'POCA 2002 in notice':       'POCA 2002' in (notice or ''),
            'notice is not None':        notice is not None,
            'does not confirm in notice':'does not confirm' in (notice or ''),
        }
        for check, passed in checks.items():
            if not passed:
                assertion_failures.append(f"[{txn}] ELEVATED RISK failed: {check}")
                print(f"  ✗ [{txn}] {check}")
            else:
                print(f"  ✓ [{txn}] {check}")

    # 3. LOW RISK must have no compliance notice
    elif tier == 'LOW RISK':
        passed = notice is None
        status = '✓' if passed else '✗'
        print(f"  {status} [{txn}] LOW RISK: compliance_notice is None")
        if not passed:
            assertion_failures.append(f"[{txn}] LOW RISK: compliance_notice should be None, got content")

    # 4. FATF references must NOT appear in the LLM-generated brief
    # (they belong in fatf_ref fields only, not the prose narrative)
    fatf_in_brief = 'FATF' in brief or 'R.10' in brief or 'R.20' in brief or 'R.15' in brief
    status = '✗' if fatf_in_brief else '✓'
    print(f"  {status} [{txn}] FATF references absent from investigation_brief")
    if fatf_in_brief:
        assertion_failures.append(f"[{txn}] FATF reference leaked into investigation_brief")

    # 5. Brief must not contain ML terminology
    ml_terms = ['SHAP', 'XGBoost', 'Random Forest', 'shap_value', 'model score',
                'feature importance', 'predict_proba']
    ml_found = [t for t in ml_terms if t.lower() in brief.lower()]
    status = '✗' if ml_found else '✓'
    print(f"  {status} [{txn}] No ML jargon in brief"
          + (f" — found: {ml_found}" if ml_found else ""))
    if ml_found:
        assertion_failures.append(f"[{txn}] ML jargon in brief: {ml_found}")

    # 6. Brief must be non-empty
    if not brief or len(brief.strip()) < 20:
        assertion_failures.append(f"[{txn}] investigation_brief is empty or too short")
        print(f"  ✗ [{txn}] Brief is empty or too short")

print(f"\n{SEPARATOR}")
if assertion_failures:
    print(f"FAILED CHECKS ({len(assertion_failures)}):")
    for f in assertion_failures:
        print(f"  - {f}")
    print("\nFix: tighten build_sar_prompt() for the failing patterns and re-run.")
else:
    print("✓ All automated checks passed.")
    print("Proceed to manual hallucination review below.")

# ---------------------------------------------------------------------------
# Manual hallucination check prompt — print for your own review
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print("MANUAL HALLUCINATION REVIEW — check each brief yourself:\n")
print("For each case: does every claim in the brief trace back to a")
print("risk-increasing SHAP factor listed above it?")
print("Specifically look for:")
print("  - Invented amounts or dollar figures")
print("  - Specific card numbers, account IDs, or names")
print("  - Legal assertions ('this is money laundering', 'fraud confirmed')")
print("  - Risk factors not present in the top_features list")
print("\nIf any of those appear: tighten the 'Do not fabricate' rule")
print("in build_sar_prompt() and add the specific pattern you saw.")

# ---------------------------------------------------------------------------
# Fill in before Day 19 checkpoint:
#
# Pipeline with LLM (avg latency):      ___ms
# Pipeline without LLM (avg latency):   ___ms
# HIGH RISK cases in sample:            ___
# ELEVATED RISK cases:                  ___
# LOW RISK cases:                       ___
# narrative_source = 'llm':             ___
# narrative_source = 'fallback':        ___
# POCA 2002 language verified:          ✓ / ✗
# Disclaimer language verified:         ✓ / ✗
# FATF leak into brief:                 ✓ (clean) / ✗ (leaked)
# ML jargon in brief:                   ✓ (clean) / ✗ (found)
# Hallucinations found:                 ___
# Prompt iterations needed:             ___
# ---------------------------------------------------------------------------

Test set: 5 fraud | 3 borderline | 2 legit
Total: 9 cases


Transaction    : 3518517
Score          : 0.0650   |   Actual: FRAUD
Risk Tier      : LOW RISK
Narrative source: llm
LLM model used : llama-3.3-70b-versatile

Risk themes    : ['Attribute Consistency', 'Identity Association']

Top SHAP factors (risk-increasing only):

  Attribute Consistency (1 contributing feature)

      [+0.01252]  M5

  Narrative:
  This signal reflects a match or mismatch between identity attributes
  associated with this transaction — such as names on the card versus the
  billing address. Inconsistencies of this type may indicate elevated identity
  verification risk warranting further review.

  FATF:
  Contributes to monitoring considerations associated with FATF R.10 (Customer
  Due Diligence). Attribute-consistency discrepancies are cited as identity-
  risk indicators in FATF typology guidance.


  Identity Association (3 contributing features)

      [+0.01094]  C6
      [+0.00762]  C2
      [+0.0

In [3]:
# ---------------------------------------------------------------------------
# Force a HIGH RISK synthetic test — verifies POCA 2002 path
# since no natural HIGH RISK cases appeared in the 100-row sample
# ---------------------------------------------------------------------------

from shap_narratives import build_shap_dict
from sar_drafter import draft_sar_narrative
import numpy as np

# Use the highest-scoring case's SHAP row, override prob to 0.85
highest_idx  = np.argmax(y_prob_test[:100])
synthetic_id = "SYNTHETIC_HIGH_RISK_TEST"

shap_dict_high = build_shap_dict(
    transaction_id  = synthetic_id,
    fraud_prob      = 0.85,           # force HIGH RISK tier
    shap_values_row = shap_values[highest_idx],
    feature_names   = feat_cols,
)
narrative_high = draft_sar_narrative(shap_dict_high)

print("=== SYNTHETIC HIGH RISK VERIFICATION ===")
print(f"Tier   : {narrative_high['risk_tier']}")
print(f"Source : {narrative_high['narrative_source']}")
print(f"\nINVESTIGATION BRIEF:\n{narrative_high['investigation_brief']}")
print(f"\nCOMPLIANCE NOTICE:\n{narrative_high['compliance_notice']}")

# Assert POCA 2002 s.330 language
notice = narrative_high['compliance_notice'] or ''
assert 'POCA 2002' in notice,                    "FAIL: POCA 2002 missing"
assert 'as soon as is practicable' in notice,    "FAIL: s.330 phrase missing"
assert 'National Crime Agency' in notice,        "FAIL: UKFIU/NCA missing"
assert 'does not confirm' in notice,             "FAIL: disclaimer missing"
assert narrative_high['risk_tier'] == 'HIGH RISK'
print("\n✓ All HIGH RISK / POCA 2002 assertions passed on synthetic case.")

=== SYNTHETIC HIGH RISK VERIFICATION ===
Tier   : HIGH RISK
Source : llm

INVESTIGATION BRIEF:
This transaction has been assessed as high risk with a fraud risk score of 0.850, indicating elevated identity verification risk due to an unusual number of addresses or entities associated with the payment card. The high volume of associated entities may suggest a heightened risk of identity linkage issues. The recommended next action is to escalate this transaction for enhanced analyst investigation, reviewing the transaction context and supporting evidence to determine the appropriate course of action.

COMPLIANCE NOTICE:
AUTOMATED ESCALATION NOTICE — FOR ANALYST REVIEW:

This transaction has been flagged at HIGH RISK tier by the automated detection system. This output represents an automated escalation for human analyst review to determine whether a disclosure to the UK Financial Intelligence Unit (UKFIU) within the National Crime Agency (NCA) is required.

Under the Proceeds of Crime Act

In [7]:
# ---------------------------------------------------------------------------
# Find a high-risk transaction from the test set
# ---------------------------------------------------------------------------

import numpy as np
df_test=pd.read_csv("../../data/processed/test_engineered.csv")
y_prob_test=joblib.load("../../models/y_prob_test.pkl")
# Create a copy so we don't modify df_test
high_risk_df = df_test.copy()
high_risk_df["fraud_score"] = y_prob_test

# Keep only transactions above the chosen threshold
high_risk_df = (
    high_risk_df[high_risk_df["fraud_score"] >= threshold]
    .sort_values("fraud_score", ascending=False)
)

print(f"High-risk transactions found: {len(high_risk_df)}\n")

# Display the top 10 highest-risk transactions
print(high_risk_df[["TransactionID", "fraud_score"]].head(10))

# ---------------------------------------------------------------------------
# Select one example for explanation/demo
# ---------------------------------------------------------------------------

example = high_risk_df.iloc[0]

print("\nExample High-Risk Transaction")
print("-" * 35)
print(f"TransactionID : {example['TransactionID']}")
print(f"Fraud Score   : {example['fraud_score']:.4f}")

# Get the corresponding index for SHAP/SAR pipeline
idx = high_risk_df.index[0]

print(f"Dataset Index : {idx}")

High-risk transactions found: 1855

       TransactionID  fraud_score
33166        3551652        0.965
38139        3556625        0.960
33161        3551647        0.960
38172        3556658        0.950
21948        3540434        0.945
21947        3540433        0.945
21949        3540435        0.940
10514        3529000        0.940
21952        3540438        0.940
57192        3575678        0.940

Example High-Risk Transaction
-----------------------------------
TransactionID : 3551652
Fraud Score   : 0.9650
Dataset Index : 33166


Testing 1 transaction(s)


IndexError: index 33166 is out of bounds for axis 0 with size 500

In [9]:
print(len(y_test), len(y_prob_test), shap_values.shape[0])

59054 59054 500


In [10]:
sample_df = df_test.iloc[:500]
match = sample_df[sample_df['TransactionID'] == 3551652]
print(match)

Empty DataFrame
Columns: [TransactionID, isFraud, TransactionDT, TransactionAmt, ProductCD, card1, card2, card3, card4, card5, card6, addr1, addr2, dist1, dist2, P_emaildomain, R_emaildomain, C1, C2, C3, C4, C5, C6, C7, C8, C9, C10, C11, C12, C13, C14, D1, D2, D3, D4, D5, D6, D7, D8, D9, D10, D11, D12, D13, D14, D15, M1, M2, M3, M4, M5, M6, M7, M8, M9, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V15, V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, V29, V30, V31, V32, V33, V34, V35, V36, V37, V38, V39, V40, V41, V42, V43, V44, V45, ...]
Index: []

[0 rows x 475 columns]


### Below code is a replication of first cell, with a larger bound of 2000 and local compute of SHAP if sample outside shap.pkl

In [10]:
# notebooks/05_sar_drafter.ipynb
# Day 18 — SAR Drafter Testing + Output Quality Review
# Run after shap_narratives.py and sar_drafter.py are confirmed complete.
# Requires: shap_values, y_prob_test, y_test, df_test, feat_cols
# from your Day 15 SHAP analysis in memory (or reload below).

import numpy as np
import pandas as pd
import json
import joblib
import os
import sys
sys.path.insert(0, os.path.abspath(".."))  # or wherever your actual sar_drafter.py lives
from shap_narratives import build_shap_dict
from sar_drafter import draft_sar_narrative

# ---------------------------------------------------------------------------
# Reload artifacts if running fresh (skip if already in memory from Day 15)
# ---------------------------------------------------------------------------
model       = joblib.load('../../models/fraud_model.pkl')
feat_cols   = joblib.load('../../models/feature_cols.pkl')
threshold   = joblib.load('../../models/threshold.pkl')
explainer   = joblib.load('../../models/shap_explainer.pkl')
shap_values = joblib.load('../../models/shap_values_sample.pkl')
y_test=joblib.load("../../models/y_test.pkl")
y_prob_test=joblib.load("../../models/y_prob_test.pkl")
df_test=pd.read_csv("../../data/processed/test_engineered.csv")
# ---------------------------------------------------------------------------
# NOTE: shap_values_sample.pkl was saved on 2000 test rows.
# y_prob_test and y_test below must index into that same 2000-row slice.
# If you saved the full test set probs separately, adjust accordingly.
# ---------------------------------------------------------------------------
# Function to help wrap the investigation brief
import textwrap

def print_brief(text, width=80):
    for para in text.split('\n'):
        print(textwrap.fill(para, width=width) if para.strip() else '')
# ---------------------------------------------------------------------------
# Grouped SHAP display helper — purely presentational, no data changes
# ---------------------------------------------------------------------------
from collections import OrderedDict
import textwrap as _tw

def _print_shap_grouped(top_features: list, narrative_width: int = 76) -> None:
    """
    Display risk-increasing SHAP features grouped by risk category.
    Narrative and FATF reference are printed once per category, not per feature.
    Sort order (descending abs_impact) is preserved within each group.
    """
    positive = [f for f in top_features if f.get('shap_value', 0) > 0]

    if not positive:
        print("  No dominant risk-increasing factors identified.")
        return

    groups: OrderedDict = OrderedDict()
    for f in positive:
        cat = f.get('category', 'Unknown')
        groups.setdefault(cat, []).append(f)

    for category, features in groups.items():
        count = len(features)
        label = f"{count} contributing feature{'s' if count > 1 else ''}"
        print(f"\n  {category} ({label})\n")

        for f in features:
            print(f"      [{f['shap_value']:+.5f}]  {f['feature']}")

        rep = features[0]
        print("\n  Narrative:")
        for line in _tw.wrap(rep['narrative'], width=narrative_width):
            print(f"  {line}")
        print("\n  FATF:")
        for line in _tw.wrap(rep['fatf_ref'], width=narrative_width):
            print(f"  {line}")
        print()


# ---------------------------------------------------------------------------
# Define test indices across the 3 expected tier categories
# ---------------------------------------------------------------------------

# 3 high-risk cases — HIGH RISK territory (score >= 0.65)
high_risk_mask    = (y_prob_test[:5000] >= 0.75)
high_risk_indices = np.where(high_risk_mask)[0][:3]

# 5 confirmed fraud cases
fraud_indices = np.where(y_test.values[:5000] == 1)[0][:5]

# 3 borderline cases — ELEVATED RISK territory (score 0.35-0.65)
# Adjust bounds if your threshold means borderline lands elsewhere
borderline_mask    = (y_prob_test[:5000] > 0.35) & (y_prob_test[:5000] < 0.65)
borderline_indices = np.where(borderline_mask)[0][:3]

# 2 legitimate low-risk cases (score < 0.15, actual = 0)
legit_mask    = (y_prob_test[:5000] < 0.15) & (y_test.values[:5000] == 0)
legit_indices = np.where(legit_mask)[0][:2]

test_indices = list(dict.fromkeys(
    list(high_risk_indices)
    + list(fraud_indices)
    + list(borderline_indices)
    + list(legit_indices)
))

print(f"Test set: {len(high_risk_indices)} high-risk | "
      f"{len(fraud_indices)} fraud | "
      f"{len(borderline_indices)} borderline | "
      f"{len(legit_indices)} legit")
print(f"Total: {len(test_indices)} cases\n")

# ---------------------------------------------------------------------------
# Run the full SHAP → narrative pipeline on each case
# ---------------------------------------------------------------------------

results = []

for idx in test_indices:
    if idx < len(shap_values):
        shap_row = shap_values[idx]                        # 1D SHAP array
    else:
        # idx falls outside the saved SHAP sample — compute it live
        row_features = df_test.iloc[[idx]][feat_cols]
        raw_shap = explainer.shap_values(row_features)

        if isinstance(raw_shap, list):
            shap_row = raw_shap[1][0]              # old SHAP: list[class][row]
        elif raw_shap.ndim == 3:
            shap_row = raw_shap[0, :, 1]            # new SHAP: (row, feature, class)
        else:
            shap_row = raw_shap[0]                  # 2D: (row, feature)

    prob     = float(y_prob_test[idx])
    txn_id   = str(df_test.iloc[idx]['TransactionID'])
    actual   = int(y_test.iloc[idx])

    shap_dict = build_shap_dict(txn_id, prob, shap_row, feat_cols)
    narrative = draft_sar_narrative(shap_dict)

    results.append({
        'txn_id':    txn_id,
        'prob':      round(prob, 4),
        'tier':      narrative['risk_tier'],
        'actual':    'FRAUD' if actual else 'LEGIT',
        'shap_dict': shap_dict,
        'narrative': narrative,
    })

# ---------------------------------------------------------------------------
# Print full output for manual checklist review
# ---------------------------------------------------------------------------

SEPARATOR = "=" * 72

for r in results:
    print(f"\n{SEPARATOR}")
    print(f"Transaction    : {r['txn_id']}")
    print(f"Score          : {r['prob']:.4f}   |   Actual: {r['actual']}")
    print(f"Risk Tier      : {r['tier']}")
    print(f"Narrative source: {r['narrative']['narrative_source']}")   # 'llm' or 'fallback'
    print(f"LLM model used : {r['narrative']['llm_model']}")

    print(f"\nRisk themes    : {r['shap_dict']['risk_themes']}")

    print(f"\nTop SHAP factors (risk-increasing only):")
    _print_shap_grouped(r['shap_dict']['top_features'])

    print(f"\nINVESTIGATION BRIEF ({r['narrative']['narrative_source'].upper()}):")
    print_brief(r['narrative']['investigation_brief'])

    if r['narrative']['compliance_notice']:
        print(f"\nCOMPLIANCE NOTICE (deterministic):")
        print(r['narrative']['compliance_notice'])
        # print(r['narrative']['compliance_notice'][:500] + "...")
    else:
        print(f"\nCOMPLIANCE NOTICE: None ({r['tier']} — not required)")

# ---------------------------------------------------------------------------
# Summary counts
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print(f"Total cases tested : {len(results)}")
print(f"HIGH RISK          : {sum(1 for r in results if r['tier'] == 'HIGH RISK')}")
print(f"ELEVATED RISK      : {sum(1 for r in results if r['tier'] == 'ELEVATED RISK')}")
print(f"LOW RISK           : {sum(1 for r in results if r['tier'] == 'LOW RISK')}")
print(f"LLM-generated      : {sum(1 for r in results if r['narrative']['narrative_source'] == 'llm')}")
print(f"Fallback briefs    : {sum(1 for r in results if r['narrative']['narrative_source'] == 'fallback')}")

# ---------------------------------------------------------------------------
# Automated assertion checks — mirrors the quality checklist
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print("AUTOMATED ASSERTION CHECKS\n")

assertion_failures = []

for r in results:
    txn = r['txn_id']
    tier = r['tier']
    notice = r['narrative']['compliance_notice']
    brief  = r['narrative']['investigation_brief']

    # 1. HIGH RISK must have POCA 2002 s.330 language in compliance_notice
    if tier == 'HIGH RISK':
        checks = {
            'POCA 2002 in notice':               'POCA 2002' in (notice or ''),
            'as soon as is practicable':         'as soon as is practicable' in (notice or ''),
            'UKFIU/NCA in notice':               'National Crime Agency' in (notice or ''),
            'does not confirm in notice':        'does not confirm' in (notice or ''),
            'notice is not None':                notice is not None,
        }
        for check, passed in checks.items():
            if not passed:
                assertion_failures.append(f"[{txn}] HIGH RISK failed: {check}")
                print(f"  ✗ [{txn}] {check}")
            else:
                print(f"  ✓ [{txn}] {check}")

    # 2. ELEVATED RISK must have analyst review language + POCA mention
    elif tier == 'ELEVATED RISK':
        checks = {
            'POCA 2002 in notice':       'POCA 2002' in (notice or ''),
            'notice is not None':        notice is not None,
            'does not confirm in notice':'does not confirm' in (notice or ''),
        }
        for check, passed in checks.items():
            if not passed:
                assertion_failures.append(f"[{txn}] ELEVATED RISK failed: {check}")
                print(f"  ✗ [{txn}] {check}")
            else:
                print(f"  ✓ [{txn}] {check}")

    # 3. LOW RISK must have no compliance notice
    elif tier == 'LOW RISK':
        passed = notice is None
        status = '✓' if passed else '✗'
        print(f"  {status} [{txn}] LOW RISK: compliance_notice is None")
        if not passed:
            assertion_failures.append(f"[{txn}] LOW RISK: compliance_notice should be None, got content")

    # 4. FATF references must NOT appear in the LLM-generated brief
    # (they belong in fatf_ref fields only, not the prose narrative)
    fatf_in_brief = 'FATF' in brief or 'R.10' in brief or 'R.20' in brief or 'R.15' in brief
    status = '✗' if fatf_in_brief else '✓'
    print(f"  {status} [{txn}] FATF references absent from investigation_brief")
    if fatf_in_brief:
        assertion_failures.append(f"[{txn}] FATF reference leaked into investigation_brief")

    # 5. Brief must not contain ML terminology
    ml_terms = ['SHAP', 'XGBoost', 'Random Forest', 'shap_value', 'model score',
                'feature importance', 'predict_proba']
    ml_found = [t for t in ml_terms if t.lower() in brief.lower()]
    status = '✗' if ml_found else '✓'
    print(f"  {status} [{txn}] No ML jargon in brief"
          + (f" — found: {ml_found}" if ml_found else ""))
    if ml_found:
        assertion_failures.append(f"[{txn}] ML jargon in brief: {ml_found}")

    # 6. Brief must be non-empty
    if not brief or len(brief.strip()) < 20:
        assertion_failures.append(f"[{txn}] investigation_brief is empty or too short")
        print(f"  ✗ [{txn}] Brief is empty or too short")

print(f"\n{SEPARATOR}")
if assertion_failures:
    print(f"FAILED CHECKS ({len(assertion_failures)}):")
    for f in assertion_failures:
        print(f"  - {f}")
    print("\nFix: tighten build_sar_prompt() for the failing patterns and re-run.")
else:
    print("✓ All automated checks passed.")
    print("Proceed to manual hallucination review below.")

# ---------------------------------------------------------------------------
# Manual hallucination check prompt — print for your own review
# ---------------------------------------------------------------------------

print(f"\n{SEPARATOR}")
print("MANUAL HALLUCINATION REVIEW — check each brief yourself:\n")
print("For each case: does every claim in the brief trace back to a")
print("risk-increasing SHAP factor listed above it?")
print("Specifically look for:")
print("  - Invented amounts or dollar figures")
print("  - Specific card numbers, account IDs, or names")
print("  - Legal assertions ('this is money laundering', 'fraud confirmed')")
print("  - Risk factors not present in the top_features list")
print("\nIf any of those appear: tighten the 'Do not fabricate' rule")
print("in build_sar_prompt() and add the specific pattern you saw.")

# ---------------------------------------------------------------------------
# Fill in before Day 19 checkpoint:
#
# Pipeline with LLM (avg latency):      ___ms
# Pipeline without LLM (avg latency):   ___ms
# HIGH RISK cases in sample:            ___
# ELEVATED RISK cases:                  ___
# LOW RISK cases:                       ___
# narrative_source = 'llm':             ___
# narrative_source = 'fallback':        ___
# POCA 2002 language verified:          ✓ / ✗
# Disclaimer language verified:         ✓ / ✗
# FATF leak into brief:                 ✓ (clean) / ✗ (leaked)
# ML jargon in brief:                   ✓ (clean) / ✗ (found)
# Hallucinations found:                 ___
# Prompt iterations needed:             ___
# ---------------------------------------------------------------------------

Test set: 3 high-risk | 5 fraud | 3 borderline | 2 legit
Total: 12 cases


Transaction    : 3519755
Score          : 0.7650   |   Actual: FRAUD
Risk Tier      : HIGH RISK
Narrative source: llm
LLM model used : llama-3.3-70b-versatile

Risk themes    : ['Identity Association']

Top SHAP factors (risk-increasing only):

  Identity Association (5 contributing features)

      [+0.11767]  C1
      [+0.05873]  C8
      [+0.04882]  C2
      [+0.04261]  C11
      [+0.03649]  C12

  Narrative:
  This signal relates to patterns of addresses or other entities associated
  with this payment card. Identity-association patterns may indicate elevated
  identity-linkage risk, informing ongoing monitoring considerations
  associated with FATF Recommendation 10.

  FATF:
  Contributes to monitoring considerations associated with FATF R.10 (Customer
  Due Diligence). Patterns involving linked identities or addresses are
  recognised as identity-association typology indicators in FATF guidance.


INVESTI